# Business Forecasting - Week 3
## Time-Series Transformations, Features, and Forecast Accuracy

This notebook accompanies the Week 3 lecture. The goal is to make the Python readable enough that you can return to it later as a reference.

### What we will do
- transform a time series when its variation grows with its level;
- compute and interpret summary statistics;
- understand autocorrelation, ACF, and PACF;
- use STL decomposition to measure trend and seasonality strength;
- check whether a remainder behaves like white noise;
- calculate forecast errors and compare two simple forecasting methods.

**Data:** upload either `Week3_Data.zip` or the CSV files individually. No data are downloaded from the internet.

## 0. Upload the data
Run the next cell once. In Google Colab, a file picker will appear. Upload the course ZIP file or the CSV files.

The code accepts both formats and places the CSV files in one data folder so the rest of the notebook can use the same paths.

In [ ]:
# Path and zipfile are used to work with folders and ZIP archives.
from pathlib import Path
import zipfile
import shutil

# This is the folder where all CSV files will be placed.
data_dir = Path("week3_data")
data_dir.mkdir(exist_ok=True)

# In Google Colab, files.upload() opens the upload window.
# Outside Colab, the notebook simply looks for files in the current folder.
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    uploaded = {}

# Save every uploaded file into the working folder.
for filename, content in uploaded.items():
    Path(filename).write_bytes(content)

# Find ZIP files in the current folder and extract them.
for zip_path in Path(".").glob("*.zip"):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)

# Copy any individually uploaded CSV files into the same data folder.
for csv_path in Path(".").glob("*.csv"):
    if csv_path.parent != data_dir:
        shutil.copy2(csv_path, data_dir / csv_path.name)

# These are the files used in this notebook.
required_files = [
    "aus_airpassengers.csv",
    "aus_production.csv",
    "us_employment.csv",
    "aus_retail.csv",
]

# Stop with a clear message if a required file is missing.
missing = [name for name in required_files if not (data_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing files: " + ", ".join(missing) +
        ". Upload Week3_Data.zip or the CSV files and run this cell again."
    )

print("Data folder ready:", data_dir.resolve())
print("Files found:")
for name in required_files:
    print(" -", name)

## 1. Import the packages
These are the main tools used in this tutorial. `pandas` handles tables, `numpy` handles numerical calculations, `matplotlib` creates graphs, and `statsmodels` provides time-series tools such as ACF, PACF, STL, and the Ljung-Box test.

In [ ]:
# NumPy: numerical calculations such as logarithms, variances, and errors.
import numpy as np

# pandas: reading CSV files, filtering rows, sorting dates, and working with columns.
import pandas as pd

# matplotlib: creating the graphs used throughout the tutorial.
import matplotlib.pyplot as plt

# plot_acf and plot_pacf: visualize autocorrelation and partial autocorrelation.
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# STL: separates a series into trend, seasonal, and remainder components.
from statsmodels.tsa.seasonal import STL

# acorr_ljungbox: tests whether a series still contains significant autocorrelation.
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.grid"] = True

## 2. Load the four datasets
We read each CSV into a pandas DataFrame. The date columns are converted to proper date values where needed.

We will use:
- **Australian air passengers** for transformations;
- **Australian production** for summary statistics and forecast errors;
- **US employment** for ACF, PACF, STL, and diagnostics;
- **Australian retail** as an additional practice dataset.

In [ ]:
# Annual Australian air-passenger data.
air = pd.read_csv(data_dir / "aus_airpassengers.csv")

# Quarterly Australian production data. parse_dates converts ds directly to dates.
production = pd.read_csv(data_dir / "aus_production.csv", parse_dates=["ds"])

# Monthly US employment data.
employment = pd.read_csv(data_dir / "us_employment.csv", parse_dates=["ds"])

# Monthly Australian retail turnover data.
retail = pd.read_csv(data_dir / "aus_retail.csv", parse_dates=["Month"])

# Sort time-based datasets so observations are in chronological order.
production = production.sort_values("ds").reset_index(drop=True)
employment = employment.sort_values("ds").reset_index(drop=True)
retail = retail.sort_values("Month").reset_index(drop=True)

print("air:", air.shape)
print("production:", production.shape)
print("employment:", employment.shape)
print("retail:", retail.shape)

# Part A - Variance-stabilizing transformations

A transformation changes the scale of a series without changing the chronological order of the observations.

A transformation is useful when the size of the fluctuations increases as the series becomes larger. In that situation, modeling the raw scale can be more difficult because the amount of variation is not stable over time.

A common solution is the **log transformation**:

\[w_t = \log(y_t)\]

Large values are compressed more than small values. This often makes changes more comparable across different levels of the series.

## 3. Look at the Australian air-passenger series
The dataset has one annual observation per year. We first plot the original series. Focus on both the long-run increase and the size of the year-to-year changes.

In [ ]:
# Plot Year on the horizontal axis and Passengers on the vertical axis.
plt.plot(air["Year"], air["Passengers"], marker="o", markersize=3)
plt.title("Australian air passengers - original scale")
plt.xlabel("Year")
plt.ylabel("Passengers")
plt.show()

### What do we see?
The series generally rises over time. The later values are much larger than the earlier values, and the absolute size of the changes also becomes larger.

This is exactly the type of situation where a variance-stabilizing transformation may help. We are not trying to remove the long-run movement; we are changing the scale so that relative changes are easier to compare.

## 4. Apply a logarithmic transformation
We create a new column rather than replacing the original data. This lets us compare the original and transformed scales directly.

In [ ]:
# np.log() applies the natural logarithm to every observation in Passengers.
# A new column is created so the original series remains available.
air["Log_Passengers"] = np.log(air["Passengers"])

# Compare the original series and its log transformation.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(air["Year"], air["Passengers"])
axes[0].set_title("Original scale")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Passengers")

axes[1].plot(air["Year"], air["Log_Passengers"])
axes[1].set_title("Log scale")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("log(Passengers)")

plt.tight_layout()
plt.show()

### Interpretation
The log transformation compresses the high values more strongly. The series still increases, but the variation is less dominated by the largest observations.

A useful way to remember the log scale is that differences on a log scale are closely related to **percentage or proportional changes** on the original scale.

# Part B - Time-series features

A **feature** is a numerical description of some characteristic of a time series. Features help us summarize many observations with a smaller set of interpretable numbers.

We will look at four groups of features:
1. summary statistics;
2. autocorrelation features;
3. STL-based trend and seasonality strengths;
4. diagnostic features related to white noise.

## 5. Summary statistics
We use the Australian quarterly production dataset. It contains several production series measured over the same quarters, including Beer, Electricity, and Gas.

The `describe()` method gives a quick numerical summary: count, mean, standard deviation, minimum, quartiles, and maximum.

In [ ]:
# Select three numerical production columns.
# Double brackets return a DataFrame containing only these columns.
selected = production[["Beer", "Electricity", "Gas"]]

# describe() summarizes the distribution of every selected variable.
summary = selected.describe().T
summary

### How to read the summary
- **mean**: average level of the series;
- **std**: typical amount of variation around the mean;
- **min / max**: observed range;
- **25%, 50%, 75%**: quartiles; the 50% value is the median.

These statistics are useful, but they do **not** tell us how observations are ordered over time. Two series can have similar means and standard deviations but completely different trend or seasonal patterns.

## 6. Distribution of quarterly beer production
A histogram shows where values are concentrated. A box plot summarizes the median, quartiles, spread, and potential extreme observations.

In [ ]:
beer = production["Beer"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Histogram: groups observations into intervals and counts how many fall in each interval.
axes[0].hist(beer, bins=16, edgecolor="white")
axes[0].set_title("Quarterly beer production - histogram")
axes[0].set_xlabel("Beer production")
axes[0].set_ylabel("Count")

# Box plot: summarizes the center and spread of the same observations.
axes[1].boxplot(beer)
axes[1].set_title("Quarterly beer production - box plot")
axes[1].set_ylabel("Beer production")

plt.tight_layout()
plt.show()

# Part C - Autocorrelation, ACF, and PACF

**Autocorrelation** measures how strongly a time series is related to its own past values.

A lag of 1 compares observations one period apart. A lag of 12 in monthly data compares observations one year apart.

Autocorrelation matters because forecasting is possible partly when the past contains useful information about the future.

## 7. Small autocorrelation example
The following ten observations are the same type of small example used in class. We can calculate lag-1 and lag-2 autocorrelation directly with pandas.

In [ ]:
# Create the small time series.
A = pd.Series([5, 8.5, 7, 9, 11, 16.2, 17, 16.8, 19, 19.2])

# autocorr(lag=1) compares A_t with A_(t-1).
r1 = A.autocorr(lag=1)

# autocorr(lag=2) compares A_t with A_(t-2).
r2 = A.autocorr(lag=2)

print(f"Lag-1 autocorrelation: {r1:.3f}")
print(f"Lag-2 autocorrelation: {r2:.3f}")

### Interpretation
Both values are strongly positive. High values tend to follow high values and low values tend to follow low values.

The lag-1 value is slightly stronger than lag 2, which is reasonable because observations closer together in time often have a stronger relationship.

## 8. Prepare monthly US retail employment
`us_employment.csv` contains many employment series. We only need the series whose `unique_id` is `Retail Trade`.

`query()` filters the rows using a readable condition. `.copy()` creates an independent DataFrame so later changes do not accidentally affect a filtered view of the original table.

In [ ]:
# query() keeps only Retail Trade observations from 1990 onward.
# copy() makes the filtered result an independent working DataFrame.
retail_emp = (
    employment
    .query("unique_id == 'Retail Trade' and ds >= '1990-01-01'")
    .copy()
    .sort_values("ds")
)

plt.plot(retail_emp["ds"], retail_emp["y"])
plt.title("US retail employment")
plt.xlabel("Date")
plt.ylabel("Persons (thousands)")
plt.show()

## 9. ACF - Autocorrelation Function
The ACF calculates autocorrelation at many lags and displays them together.

For monthly data, pay special attention to lags such as 12, 24, and 36. Peaks around these lags often indicate annual seasonality. A slow decline across many lags can also be a sign of trend or strong persistence.

In [ ]:
# plot_acf() calculates and plots autocorrelation for many lags.
# lags=48 asks for the first four years of monthly lags.
plot_acf(retail_emp["y"], lags=48, zero=False)
plt.title("ACF of US retail employment")
plt.xlabel("Lag (months)")
plt.show()

### What the ACF is telling us
The correlations remain large across many lags, which reflects strong persistence in the series. Seasonal structure can also appear around multiples of 12 months.

Remember that an ACF is describing the **combined behavior** of the series: trend, seasonality, cycles, and other persistence can all contribute to the observed correlations.

## 10. PACF - Partial Autocorrelation Function
The PACF asks a more specific question: what is the relationship between the current observation and a particular lag **after accounting for shorter lags in between**?

This becomes especially useful later when we study autoregressive forecasting models.

In [ ]:
# plot_pacf() shows the direct relationship with each lag after controlling for intermediate lags.
plot_pacf(retail_emp["y"], lags=36, method="ywm", zero=False)
plt.title("PACF of US retail employment")
plt.xlabel("Lag (months)")
plt.show()

# Part D - STL features

STL decomposition separates a series into:

\[y_t = T_t + S_t + R_t\]

where:
- \(T_t\) is the trend component;
- \(S_t\) is the seasonal component;
- \(R_t\) is the remainder.

Once the series is decomposed, we can turn the visual ideas of “strong trend” and “strong seasonality” into numerical features.

## 11. Decompose US retail employment with STL
This is monthly data, so `period=12` tells STL that a seasonal cycle contains 12 observations. `robust=True` reduces the influence of unusually large observations when estimating the components.

In [ ]:
# STL receives the values of the employment series.
# period=12 represents one year of monthly observations.
stl = STL(retail_emp["y"], period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
axes[0].plot(retail_emp["ds"], retail_emp["y"])
axes[0].set_ylabel("Observed")
axes[1].plot(retail_emp["ds"], stl.trend)
axes[1].set_ylabel("Trend")
axes[2].plot(retail_emp["ds"], stl.seasonal)
axes[2].set_ylabel("Seasonal")
axes[3].plot(retail_emp["ds"], stl.resid)
axes[3].set_ylabel("Remainder")
axes[3].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 12. Trend strength
A strong trend means that the trend-plus-remainder varies much more than the remainder alone.

A commonly used measure is:

\[F_T = \max\left(0, 1 - rac{Var(R_t)}{Var(T_t + R_t)}ight)\]

Values near 1 indicate strong trend. Values near 0 indicate little trend.

In [ ]:
# Save the three STL components as short variable names.
T = stl.trend
S = stl.seasonal
R = stl.resid

# np.var(..., ddof=1) calculates sample variance.
trend_strength = max(
    0,
    1 - np.var(R, ddof=1) / np.var(T + R, ddof=1)
)

print(f"Trend strength: {trend_strength:.3f}")

## 13. Seasonality strength
The same idea can measure seasonality:

\[F_S = \max\left(0, 1 - rac{Var(R_t)}{Var(S_t + R_t)}ight)\]

If removing seasonality leaves a much smaller remainder, the seasonal component was important and the strength value will be close to 1.

In [ ]:
season_strength = max(
    0,
    1 - np.var(R, ddof=1) / np.var(S + R, ddof=1)
)

print(f"Seasonality strength: {season_strength:.3f}")

### Interpretation for this series
For US retail employment, both measures are close to 1. The series has a very strong long-run structure and strong recurring seasonality.

Features like these become especially useful when an organization manages hundreds or thousands of time series and needs a systematic way to identify which series are strongly trending or seasonal.

## 14. Is the remainder white noise?
After removing trend and seasonality, we hope the remainder contains little predictable structure.

The **Ljung-Box test** checks whether a group of autocorrelations is jointly equal to zero.

- Null hypothesis: the tested autocorrelations are zero.
- Small p-value: evidence that autocorrelation remains.

In [ ]:
# Test the STL remainder through lag 12.
# return_df=True returns the statistic and p-value in a DataFrame.
ljung_box = acorr_ljungbox(pd.Series(R).dropna(), lags=[12], return_df=True)
ljung_box

### Important point
A decomposition can be useful even when the remainder is not perfect white noise. If significant autocorrelation remains, there may still be structure available for a forecasting model to use.

So decomposition is not the final forecasting model by itself; it is also a diagnostic tool that helps us understand what remains after major components are removed.

# Part E - Forecast errors and accuracy

A forecast error is defined as:

\[e_t = y_t - \hat{y}_t\]

where \(y_t\) is the actual value and \(\hat{y}_t\) is the forecast.

- Positive error: actual was higher than forecast, so the model **under-forecasted**.
- Negative error: actual was lower than forecast, so the model **over-forecasted**.

## 15. Create a train/test split using quarterly beer production
We keep the final eight quarters as a test period. The model will not use these observations when forming its forecast. This imitates the real forecasting situation: predict unseen future data, then compare predictions with what actually occurred.

In [ ]:
# Keep only the date and Beer columns and remove missing values.
beer_data = production[["ds", "Beer"]].dropna().copy().sort_values("ds").reset_index(drop=True)

# The final 8 quarters become the test set.
test = beer_data.tail(8).copy()

# Everything before the final 8 quarters becomes training data.
train = beer_data.iloc[:-8].copy()

print("Training observations:", len(train))
print("Test observations:", len(test))

## 16. Baseline 1 - Naive forecast
A naive forecast uses the most recent training value as the forecast for every future period. It is simple, but simple baselines are important because a more complicated model should ideally improve on them.

In [ ]:
# iloc[-1] means the final row in the training data.
last_training_value = train["Beer"].iloc[-1]

# np.repeat() repeats that last value once for each test observation.
naive_forecast = np.repeat(last_training_value, len(test)).astype(float)

print("Last training value:", last_training_value)
print("Naive forecasts:", naive_forecast)

## 17. Baseline 2 - Seasonal-naive forecast
Quarterly beer production is seasonal. A seasonal-naive forecast uses the value from the same quarter one year earlier.

For quarterly data, one year corresponds to four quarters.

In [ ]:
# Put dates in the index so we can retrieve a value using its date.
beer_by_date = beer_data.set_index("ds")["Beer"]

seasonal_naive_forecast = []

for current_date in test["ds"]:
    # DateOffset(years=1) moves exactly one year back.
    previous_year_date = current_date - pd.DateOffset(years=1)

    # .loc[] retrieves the beer value observed on that earlier date.
    seasonal_naive_forecast.append(beer_by_date.loc[previous_year_date])

# Convert the list to a NumPy array for calculations.
seasonal_naive_forecast = np.array(seasonal_naive_forecast, dtype=float)

seasonal_naive_forecast

## 18. Calculate forecast errors
We calculate errors for the seasonal-naive method first. The sign of the error tells us the direction of the miss.

In [ ]:
actual = test["Beer"].to_numpy(dtype=float)

# Error = Actual - Forecast.
errors = actual - seasonal_naive_forecast

error_table = pd.DataFrame({
    "Date": test["ds"].to_numpy(),
    "Actual": actual,
    "Forecast": seasonal_naive_forecast,
    "Error": errors,
})

error_table

## 19. MAE / MAD - Mean Absolute Error
MAE averages the absolute size of the errors:

\[MAE = rac{1}{n}\sum |e_t|\]

Some courses and organizations call the same idea MAD, Mean Absolute Deviation. Because the absolute value is used, positive and negative errors cannot cancel each other.

In [ ]:
mae = np.mean(np.abs(errors))
print(f"MAE: {mae:.2f}")

## 20. MSE and RMSE
MSE squares the errors before averaging them:

\[MSE = rac{1}{n}\sum e_t^2\]

Large errors receive much more weight. RMSE takes the square root of MSE, returning the measure to the same unit as the original series.

In [ ]:
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

## 21. MAPE - Mean Absolute Percentage Error
MAPE expresses the average absolute error as a percentage of the actual value:

\[MAPE = rac{100}{n}\sum \left|rac{e_t}{y_t}ight|\]

Its advantage is interpretability across different units. Its limitation is that it becomes problematic when actual values are zero or very close to zero.

In [ ]:
mape = np.mean(np.abs(errors / actual)) * 100
print(f"MAPE: {mape:.2f}%")

## 22. Compare the two baseline forecasting methods
A useful accuracy measure should be calculated on the same test period for every model. Lower values indicate smaller forecast errors.

In [ ]:
def accuracy_metrics(actual, forecast):
    """Return four common forecast-accuracy measures."""
    error = actual - forecast
    return {
        "MAE": np.mean(np.abs(error)),
        "MSE": np.mean(error ** 2),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "MAPE": np.mean(np.abs(error / actual)) * 100,
    }

comparison = pd.DataFrame({
    "Naive": accuracy_metrics(actual, naive_forecast),
    "Seasonal naive": accuracy_metrics(actual, seasonal_naive_forecast),
}).T

comparison.round(2)

### Interpretation
For this quarterly beer series and this test period, the seasonal-naive method has substantially smaller errors than the ordinary naive method.

That is a useful forecasting lesson: **a baseline that respects an obvious pattern in the data can outperform a simpler baseline by a large margin.**

We are not claiming that seasonal naive is the best possible model. It is a benchmark that future models should try to beat.

# Quick reference - code worth remembering

These are the patterns you will reuse frequently in later tutorials.

```python
# Read a CSV file.
df = pd.read_csv("my_file.csv")

# Convert a date column.
df["Date"] = pd.to_datetime(df["Date"])

# Filter rows with query().
# Example: keep one series and dates after a starting point.
filtered = df.query("Category == 'Retail' and Date >= '2020-01-01'").copy()

# Summary statistics.
summary = df["Value"].describe()

# Autocorrelation at one lag.
r1 = df["Value"].autocorr(lag=1)

# ACF and PACF graphs.
plot_acf(df["Value"], lags=24)
plot_pacf(df["Value"], lags=24, method="ywm")

# STL decomposition for monthly data.
stl_result = STL(df["Value"], period=12, robust=True).fit()

# Forecast error.
error = actual - forecast

# Mean absolute error.
mae = np.mean(np.abs(error))
```

# Week 3 takeaways
- Transformations can make changing variance easier to model.
- Summary statistics describe the distribution, but not the order of observations through time.
- ACF and PACF reveal relationships between current and past observations.
- STL creates trend, seasonal, and remainder components that can also be summarized as features.
- A good remainder should be checked for remaining autocorrelation.
- Forecast accuracy must be measured on data that were not used to build the forecast.
- MAE, MSE/RMSE, and MAPE answer slightly different questions, so always understand the metric you are reporting.